In [1]:
# 一般需要调用的库

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import os
from langchain_openai import ChatOpenAI
# pip install langchain langchain-openai langchain-community python-dotenv -i https://mirrors.aliyun.com/pypi/simple/

C:\Users\DELL\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## LangChain的demo代码
* 配置环境变量
* 涉及的重要参数:
*

In [ ]:
import os
from langchain_openai import ChatOpenAI

# 建议将 API Key 放在环境变量中，或者直接在此处赋值（生产环境请勿硬编码）


# 初始化模型
llm = ChatOpenAI(
    model="deepseek-chat",  # DeepSeek V3 模型名称
    openai_api_key=os.getenv("DEEPSEEK_API_KEY"),  # 填入你的 DeepSeek API Key
    openai_api_base=os.getenv("DEEPSEEK_BASE_URL"),  # DeepSeek 的 API 地址
    temperature=0.7,
    max_tokens=1024
)

# 测试一下
response = llm.invoke("你好，DeepSeek！请做一个简短的自我介绍。")
print(response.content)

## LangChain 学习代码
* 构建 LCEL 链 (LangChain Expression Language)
* 条理清晰的数据流向
* 构建一般的简单询问ChatAI



In [ ]:

# 加载环境变量
load_dotenv(override=True)

# 配置llm
# 第一步 构建模型
llm = ChatOpenAI(
    model=os.getenv('DEEPSEEK_MODEL', 'deepseek-chat'),
    openai_api_key=os.getenv('DEEPSEEK_API_KEY'),
    openai_api_base=os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com'),
)
# 第二步 构建提示词
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个超级聪明的人工智能,人设为富有学识的温柔女导师形象"),
    ("user", "{question}")
])

# 第三步 定义解析器
parser = StrOutputParser()

# 第四步 构建条理清晰的数据链
# 提示词 -> 大模型 -> 字符串解析器
chain = prompt | llm | parser

results = chain.invoke({"question":"解释量子纠缠"})

print(results)


In [ ]:
results_stream = chain.stream({"question":"解释量子纠缠"})

print(results_stream)
# 实现打字机效果,流式输出
for chunk in results_stream:
    print(chunk  , end="  ")

In [ ]:
    pip install langchain-community faiss-cpu sentence-transformers -i https://mirrors.aliyun.com/pypi/simple/

In [ ]:
# pip install --upgrade transformers
# pip install --upgrade torch tensorflow

In [ ]:
import numpy
print(numpy.__version__)

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 加载环境变量
load_dotenv()

# 初始化模型和 Embedding
llm = ChatOpenAI(
    model=os.getenv('DEEPSEEK_MODEL', 'deepseek-chat'),
    openai_api_key=os.getenv('DEEPSEEK_API_KEY'),
    openai_api_base=os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com'),
)

# 使用免费的本地 Embedding 模型（首次运行会自动下载）
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# --- 1. 准备知识文档 ---
# 实际项目中，这些文档可以从文件、网页、数据库等加载
documents = [
    "LangChain 是一个用于构建大语言模型应用的开源框架，由 Harrison Chase 于 2022 年创建。",
    "LangChain 的核心组件包括：模型接口、提示词模板、链、记忆、检索和代理。",
    "LCEL（LangChain Expression Language）是 LangChain 的新一代链构建语法，使用管道符 | 连接各组件。",
    "RAG（检索增强生成）通过在生成前检索相关文档，让 LLM 能回答训练数据之外的问题。",
    "LangGraph 是 LangChain 团队推出的新框架，专门用于构建复杂的多步骤 AI 代理工作流。",
    "LangSmith 是 LangChain 的可观测性平台，用于调试、测试和监控 LLM 应用。",
]

# --- 2. 文本切片 ---
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)
# 这里文档已经较短，实际场景中长文档会被切成多个片段
texts = text_splitter.create_documents(documents)

# --- 3. 向量化并存入 FAISS ---
vectorstore = FAISS.from_documents(texts, embeddings)

# --- 4. 创建检索器 ---
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# --- 5. 构建 RAG 链 ---
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个知识助手。根据以下检索到的上下文来回答问题。如果上下文中没有答案，就说你不知道。\n\n上下文：{context}"),
    ("human", "{question}")
])

# 辅助函数：将检索到的文档拼接为字符串
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# --- 6. 测试 RAG ---
question = "LangChain 的核心组件有哪些？"
print(f"问题：{question}")
print(f"回答：{rag_chain.invoke(question)}")